### **Dataset Overview**
1. **Load and inspect the dataset (Essential)**
    - Tasks: ``shape``, ``head``, ``info``
    - Purpose: understand available fields (text, timestamp, user, location, label, etc.).

In [4]:
import pandas as pd
import os

In [5]:
# Loading the training dataset from the local project directory
data_path = os.path.join('data', 'disaster_tweets', 'train.csv')
disaster_tweets_df = pd.read_csv(data_path)

In [6]:
# Checking the dataset's basic dimensions (total rows and columns)
print("Dataset shape: ", disaster_tweets_df.shape)

Dataset shape:  (7613, 5)


In [7]:
# Previewing the first 10 rows to get a feel for the raw tweet content and labels
disaster_tweets_df.head(10)

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
5,8,NaN,NaN,#RockyFire Update => California Hwy. 20 closed...,1
6,10,NaN,NaN,#flood #disaster Heavy rain causes flash flood...,1
7,13,NaN,NaN,I'm on top of the hill and I can see a fire in...,1
8,14,NaN,NaN,There's an emergency evacuation happening now ...,1
9,15,NaN,NaN,I'm afraid that the tornado is coming to our a...,1


We have in the dataset the following Data:
#####  **Key Feature Definitions**

- **id**: A unique identifier assigned to each tweet.
- **keyword**: A specific disaster-related term associated with the tweet (may be null).
- **location**: The user-reported location from which the tweet originated (may be null).
- **text**: The full content of the tweet.
- **target**: Our ground truth label; 1 indicates a tweet about a real disaster, while 0 indicates otherwise.

In [8]:
# Inspecting metadata: data types, non-null counts, and memory consumption
disaster_tweets_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        7613 non-null   int64
 1   keyword   7552 non-null   str  
 2   location  5080 non-null   str  
 3   text      7613 non-null   str  
 4   target    7613 non-null   int64
dtypes: int64(2), str(3)
memory usage: 1.2 MB


2. **Check data types and missing values (Essential)**
    - Tasks: ``dtypes``, ``isnull()``.``sum()``
    - Purpose: identify cleaning and preprocessing needs.

In [9]:
# dtypes is an attribute of a DataFrame
disaster_tweets_df.dtypes

id          int64
keyword       str
location      str
text          str
target      int64
dtype: object

In [10]:
# Return a boolean same-sized object indicating if the values are NA. NA values, such as None or numpy.NaN, gets mapped to True values.
disaster_tweets_df.isnull()

,id,keyword,location,text,target
0,False,True,True,False,False
1,False,True,True,False,False
2,False,True,True,False,False
3,False,True,True,False,False
4,False,True,True,False,False
...,...,...,...,...,...
7608,False,True,True,False,False
7609,False,True,True,False,False
7610,False,True,True,False,False
7611,False,True,True,False,False


In [11]:
# Apply a sum for each column with all data. Obviously a sum operation defined depend of data type
disaster_tweets_df.sum()

id                                                   41429450
keyword     ablazeablazeablazeablazeablazeablazeablazeabla...
location    BirminghamEst. September 2012 - BristolAFRICAP...
text        Our Deeds are the Reason of this #earthquake M...
target                                                   3271
dtype: object

3. **Cardinality and unique values**
    - Tasks: number of unique users, locations, hashtags, labels
    - Purpose: detect dominant users or rare categories.

In [12]:
import re

In [13]:
disaster_tweets_df['id'].value_counts().shape

(7613,)

In [14]:
# I take only one column of the DataFrame accesing with the name of the column, and apply the value_count() attribute and take the attribute shape for obtain the total number of records
# Additionally the value_counts() attribute, take the unique values of the column and count the number of these values
print("Number of unique users", disaster_tweets_df['id'].value_counts().shape[0])
print("Number of locations", disaster_tweets_df['location'].value_counts().shape[0])

""" For take the information about the number of unique hashtags, we have to process the data first. First we must take a copy of the text column since DataFrame. 
With text DataSeries, we iterate though the serie and use regultar expresions to find all ocurrences of words that begin with #. 
"""
text_disaster = disaster_tweets_df['text']
number_of_hastags = 0
for text in text_disaster:
    number_of_hastags += len(re.findall(r'#\w+', text))

print(f"Number of hashtags: ", number_of_hastags)
print("Number of labels: ", disaster_tweets_df['target'].value_counts().shape[0])

Number of unique users 7613
Number of locations 3341
Number of hashtags:  3330
Number of labels:  2


### **Text Quality and Cleaning**

4. **Duplicate tweets and retweets (Essential)**
    - Tasks: detect duplicate text, flag/remove retweets (RT @)
    - Purpose: prevent bias from highly repeated content.


In [15]:
# Define an visually options for outputs in notebook cells for DataFrames
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 50) 

In [16]:
"""
We take the DataFrame and apply a filter of where values that are True when applying the duplicated attribute.
The parameters do the following: 
- subset: Selects an specific column of the dataset to detect duplicates
- keep: Maintain all duplicated values (We apply this because we want to identify that duplicated records with different targets)
"""
duplicated = disaster_tweets_df[disaster_tweets_df.duplicated(subset=['text'], keep=False)]
duplicated

,id,keyword,location,text,target
40,59,ablaze,Live On Webcam,Check these out: http://t.co/rOI2NSmEJJ http://t.co/3Tj8ZjiN21 http://t.co/YDUiXEfIpE http://t.co/LxTjc87KLS #nsfw,0
48,68,ablaze,Live On Webcam,Check these out: http://t.co/rOI2NSmEJJ http://t.co/3Tj8ZjiN21 http://t.co/YDUiXEfIpE http://t.co/LxTjc87KLS #nsfw,0
106,156,aftershock,US,320 [IR] ICEMOON [AFTERSHOCK] | http://t.co/vAM5POdGyw | @djicemoon | #Dubstep #TrapMusic #DnB #EDM #Dance #IcesÛ_ http://t.co/zEVakJaPcz,0
115,165,aftershock,US,320 [IR] ICEMOON [AFTERSHOCK] | http://t.co/vAM5POdGyw | @djicemoon | #Dubstep #TrapMusic #DnB #EDM #Dance #IcesÛ_ http://t.co/zEVakJaPcz,0
118,171,aftershock,Switzerland,320 [IR] ICEMOON [AFTERSHOCK] | http://t.co/THyzOMVWU0 | @djicemoon | #Dubstep #TrapMusic #DnB #EDM #Dance #IcesÛ_ http://t.co/83jOO0xk29,0
...,...,...,...,...,...
7600,10855,NaN,NaN,Evacuation order lifted for town of Roosevelt: http://t.co/EDyfo6E2PU http://t.co/M5KxLPKFA1,1
7607,10867,NaN,NaN,#stormchase Violent Record Breaking EF-5 El Reno Oklahoma Tornado Nearly Runs Over ... - http://t.co/3SICroAaNz http://t.co/I27Oa0HISp,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control wild fires in California even in the Northern part of the state. Very troubling.,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. http://t.co/zDtoyd8EbJ,1


In [17]:
"""
Investigating Label Inconsistencies:
Some tweets have identical text but carry different target labels. We'll group them 
by their content and filter for cases where the number of unique targets is greater than 1.
This helps us identify and handle 'noisy' data that could confuse our future model.
"""

duplicated_focus_target = duplicated.groupby('text', as_index=False)['target'].nunique()
duplicated_with_multiple_targets = duplicated_focus_target[duplicated_focus_target['target'] > 1]
duplicated_with_multiple_targets

,text,target
0,#Allah describes piling up #wealth thinking it would last #forever as the description of the people of #Hellfire in Surah Humaza. #Reflect,2
7,#foodscare #offers2go #NestleIndia slips into loss after #Magginoodle #ban unsafe and hazardous for #humanconsumption,2
11,.POTUS #StrategicPatience is a strategy for #Genocide; refugees; IDP Internally displaced people; horror; etc. https://t.co/rqWuoy1fm4,2
24,CLEARED:incident with injury:I-495 inner loop Exit 31 - MD 97/Georgia Ave Silver Spring,2
25,Caution: breathing may be hazardous to your health.,2
32,He came to a land which was engulfed in tribal war and turned it into a land of peace i.e. Madinah. #ProphetMuhammad #islam,2
33,Hellfire is surrounded by desires so be careful and donÛªt let your desires control you! #Afterlife,2
35,Hellfire! We donÛªt even want to think about it or mention it so letÛªs not do anything that leads to it #islam!,2
37,I Pledge Allegiance To The P.O.P.E. And The Burning Buildings of Epic City. ??????,2
39,In #islam saving a person is equal in reward to saving all humans! Islam is the opposite of terrorism!,2


In [18]:
# Removing records with conflicting labels to ensure a cleaner, more consistent training signal.
not_duplicate_text = disaster_tweets_df[~disaster_tweets_df['text'].isin(duplicated_with_multiple_targets)]
not_duplicate_text

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter in place orders are expected,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation orders in California",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as smoke from #wildfires pours into a school,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse into nearby homes http://t.co/STfMbbZFB5,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control wild fires in California even in the Northern part of the state. Very troubling.,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. http://t.co/zDtoyd8EbJ,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided with a car in Little Portugal. E-bike rider suffered serious non-life threatening injuries.,1


In [19]:
# Drop the records that are Retweets using the string method startswith, to check if the text have RT or @ in the beggining of the text
disaster_tweets_df_cleaned_duplicated = not_duplicate_text
disaster_tweets_df_cleaned_RT = disaster_tweets_df_cleaned_duplicated[~(disaster_tweets_df_cleaned_duplicated['text'].str.startswith('RT') | disaster_tweets_df_cleaned['text'].str.startswith('@'))]
disaster_tweets_df_cleaned_RT.sample(10)

NameError: name 'disaster_tweets_df_cleaned' is not defined

5. **Missing or empty text entries (Essential)**
    - Tasks: remove or impute empty tweets.


In [ ]:
# Counting tweets with a character length of exactly zero
if disaster_tweets_df_cleaned_RT[disaster_tweets_df_cleaned_RT['text'].str.len() == 0].shape[0] == 0:
    print("The number of empty tweets are 0")
else:
    print("We have empty tweets")

The number of empty tweets are 0


6. **Tweet length analysis (Essential)**
    - Tasks: character length, token length distributions
    - Purpose: identify outliers and truncation risks.

In [ ]:
"""
We take the legth of the characters and show in ascending order to know the records with lower character length.
"""
disaster_tweets_df_length_tokens = disaster_tweets_df_cleaned_RT.copy()
disaster_tweets_df_length_tokens['character_length'] = disaster_tweets_df_length_tokens['text'].str.len()
disaster_tweets_df_length_tokens.sort_values(by="character_length", ascending=True)

,id,keyword,location,text,target,character_length
1882,2703,crushed,NaN,Crushed,0,7
4890,6962,massacre,NaN,Bad day,0,7
5115,7295,nuclear%20reactor,NaN,Err:509,0,7
4971,7088,meltdown,NaN,Meltdown,0,8
3670,5224,fatality,Rafael castillo,fatality,0,8
...,...,...,...,...,...,...
635,919,bioterrorism,NaN,@cspanwj If 90BLKs&amp;8WHTs colluded 2 take WHT F @USAgov AUTH Hostage&amp;2 make her look BLK w/Bioterrorism&amp;use her lgl/org IDis ID still hers?,1,150
633,915,bioterrorism,NaN,@HowardU If 90BLKs&amp;8WHTs colluded 2 take WHT F @USAgov AUTH Hostage&amp;2 make her look BLK w/Bioterrorism&amp;use her lgl/org IDis ID still hers?,1,150
614,885,bioterrorism,NaN,@CAgov If 90BLKs&amp;8WHTs colluded 2 take WHT F @USAgov AUTH Hostage&amp;2 make her look BLK w/Bioterrorism&amp;use her lgl/org IDis ID still hers?@VP,1,151
4801,6833,loud%20bang,london essex england uk,It's was about 2:30 in the morning&amp;I went downstairs to watch some telly&amp;I accidentally made a loud bang&amp;my dad(who has a broken leg)walked-,0,152


In [ ]:
# Computing word counts (tokens) by splitting the text on whitespace
disaster_tweets_df_length_tokens['tokens_length'] = disaster_tweets_df_length_tokens['text'].apply(lambda x: len(x.split()))
disaster_tweets_df_length_tokens.sort_values(by="tokens_length", ascending=True)

,id,keyword,location,text,target,character_length,tokens_length
24,36,NaN,NaN,LOOOOOOL,0,8,1
3670,5224,fatality,Rafael castillo,fatality,0,8,1
3667,5221,fatality,Nairobi,Fatality!,0,9,1
6705,9605,thunder,NaN,Thunder???,0,10,1
1882,2703,crushed,NaN,Crushed,0,7,1
...,...,...,...,...,...,...,...
6005,8577,screams,"Melbourne, Australia.",@OllyMursAus I do feel sorry for him! He is not a piece of meat! He is a nice guy... People don't need to rush him and screams in his face!,1,139,30
1947,2799,curfew,Miami,In the beginning of summer my mom made my curfew 1 now it's back to 12 and I can never go out and she wonders why I'm always at home,0,132,30
4432,6305,hostage,NaN,When u get mugged with ur gf u come up with the best excuses not to look like a bitch 'I wanted to fight but what if he held u hostage?',0,136,31
954,1381,body%20bag,NaN,If you have a son or a daughter would you like to see them going to a war with Iran and come back in a body bag? Let the #Republicans know,0,138,31


7. **URLs, mentions, hashtags, emojis extraction**
    - Tasks: extract and count each element separately
    - Purpose: these are often informative features.

In [ ]:
"""
We apply regular expresions to determine if the record contain an url, mentions, hashtags and emojis and how many for each of one.
"""
disaster_tweets_df_cleaned_RT = disaster_tweets_df_length_tokens
disaster_tweets_df_cleaned_RT['has_url'] = disaster_tweets_df_cleaned_RT['text'].str.contains(r'http?s//:S+', regex=True)
disaster_tweets_df_cleaned_RT.sample(5)

,id,keyword,location,text,target,character_length,tokens_length,has_url
4945,7048,meltdown,NaN,Deepak Chopra's EPIC Twitter Meltdown http://t.co/ethgAGPy5G,0,60,6,False
6299,9000,stretcher,Austin/Los Angeles,Stretcher brought out for Vampiro. Cut to commercial isn't a good sign. #UltimaLucha #LuchaUnderground,0,102,14,False
2692,3861,detonation,NaN,Ignition Knock (Detonation) Sensor-Senso Standard KS94 http://t.co/dY1erSDcRh http://t.co/m4cPmxmuRK,1,100,8,False
6717,9619,thunderstorm,Sydney,Beautiful lightning as seen from plane window http://t.co/5CwUyLnFUm http://t.co/1tyYqFz13D,0,91,9,False
3078,4415,electrocute,NaN,Electric vs Gas brewing (not wanting to electrocute myself) question http://t.co/26oo0fcL53,0,91,11,False


In [ ]:
disaster_tweets_df_cleaned_RT['url_count'] = disaster_tweets_df_cleaned_RT['text'].str.findall(r'http?s//:S+')
disaster_tweets_df_cleaned_RT['url_count'] = disaster_tweets_df_cleaned_RT['url_count'].apply(len)
disaster_tweets_df_cleaned_RT.sample(5)

,id,keyword,location,text,target,character_length,tokens_length,has_url,url_count
3425,4898,explode,they/them,tagged by @attackonstiles \n\nmillions\na-punk\nhang em high\nalpha dog\nyeah boy and doll face\nlittle white lies\nexplode http://t.co/lAtsSUo4wS,0,138,20,False,0
5231,7477,obliteration,NaN,I added a video to a @YouTube playlist http://t.co/1vjAlJA1SX GTA 5 Funny Moments - 'OBLITERATION!' (GTA 5 Online Funny Moments),0,128,20,False,0
4612,6555,injury,NaN,@AdamRubinESPN Familia: arm injury or head case?,1,48,7,False,0
6099,8709,sinking,"Fountain Valley, CA",Lying Clinton sinking! Donald Trump singing: Let's Make America Great Again! https://t.co/zv60cHjclF,0,100,12,False,0
5847,8355,ruin,NaN,I understand you wanting to hang out with your guy friends I'll give you your space but don't ruin my trust with you.,0,117,23,False,0


In [ ]:
disaster_tweets_df_cleaned_RT['has_mentions'] = disaster_tweets_df_cleaned_RT['text'].str.contains(r'@\w+', regex=True)
disaster_tweets_df_cleaned_RT['mentions_count'] = disaster_tweets_df_cleaned_RT['text'].str.findall(r'@\w+')
disaster_tweets_df_cleaned_RT['mentions_count'] = disaster_tweets_df_cleaned_RT['mentions_count'].apply(len)
disaster_tweets_df_cleaned_RT.sample(10)

,id,keyword,location,text,target,character_length,tokens_length,has_url,url_count,has_mentions,mentions_count
4274,6072,heat%20wave,"Oklahoma City, OK",Longest Streak of Triple-Digit Heat Since 2013 Forecast in Dallas: An unrelenting and dangerous heat wave will... http://t.co/s4Srgrmqcz,1,136,18,False,0,False,0
5832,8332,rubble,California,China's Stock Market Crash: Are There Gems In The Rubble? http://t.co/Ox3qb15LWQ | https://t.co/8u07FoqjzW http://t.co/tg5fQc8zEY,0,129,14,False,0,False,0
452,655,attack,NaN,ISIL claims suicide bombing at Saudi mosque that killed at least 15 http://t.co/Y8IcF89H6w http://t.co/t9MSnZV1Kb,1,113,14,False,0,False,0
630,907,bioterrorism,NaN,To fight bioterrorism sir.,1,26,4,False,0,False,0
5563,7938,rainstorm,"Pennsylvania, USA",RAIN RAIN GO AWAY... A soaker is on the way \n------&gt; http://t.co/jQFcY9GuqV &lt;----- http://t.co/tN65puhfhw,0,111,14,False,0,False,0
978,1415,body%20bag,NaN,#handbag #fashion #style http://t.co/hPd3SNM6oy Vintage Coach Purse Camera Bag Cross Body #9973\n\n$16.99 (0 Bids)\nÛ_ http://t.co/GSmdDmu9Pu,0,139,17,False,0,False,0
4383,6226,hijacker,NaN,@JagexHelpDibi update JAG enabled but a hijacker has access might be what I was looking for. Fingers crossed.,0,109,18,False,0,True,1
5304,7577,outbreak,Akure city in ondo state,Families to sue over Legionnaires: More than 40 families affected by the fatal outbreak of Legionnaires' disea... http://t.co/mNsy1QR7bq,1,136,18,False,0,False,0
5881,8400,sandstorm,USA,Watch This Airport Get Swallowed Up By A Sandstorm In Under A Minute http://t.co/brctMNybjy,1,91,14,False,0,False,0
5020,7160,mudslide,Birmingham & Bristol,'It looks like a mudslide' poor thing! ?? #greatbritishbakeoff,1,62,9,False,0,False,0


In [ ]:
disaster_tweets_df_cleaned_RT['has_hashtags'] = disaster_tweets_df_cleaned_RT['text'].str.contains(r'#\w+', regex=True)
disaster_tweets_df_cleaned_RT['hashtags_count'] = disaster_tweets_df_cleaned_RT['text'].str.findall(r'#\w+')
disaster_tweets_df_cleaned_RT['hashtags_count'] = disaster_tweets_df_cleaned_RT['hashtags_count'].apply(len)
disaster_tweets_df_cleaned_RT.sample(5)

,id,keyword,location,text,target,character_length,tokens_length,has_url,url_count,has_mentions,mentions_count,has_hashtags,hashtags_count
1682,2427,collide,Michigan,@mattcohen4fake Gamma Ray January Worlds Collide She Waits Be Me Wave Past Perfect Reunion Lucky Cool If I Come Over Hot Times...,0,129,22,False,0,True,1,False,0
7136,10222,volcano,"Hawaii, USA",USGS EQ: M 1.9 - 5km S of Volcano Hawaii: Time2015-08-06 01:04:01 UTC2015-08-05 15:04:01 -10:00 a... http://t.co/3rrGHT4ewp #EarthQuake,1,135,18,False,0,False,0,True,1
2006,2882,damage,"Pontevedra, Galicia",#NP Metallica - Damage Inc,0,26,5,False,0,False,0,True,1
3757,5338,fire,å_,WCW @catsandsyrup THA BITCH IS FIRE,0,35,6,False,0,True,1,False,0
7301,10448,wild%20fires,"Fort Walton Beach, FL",Firefighters Headed To California To Fight Wild Fires http://t.co/J2PYkYo0EN,1,76,9,False,0,False,0,False,0


In [ ]:
import emoji
def find_emojis(text):
    return [c for c in text if emoji.is_emoji(c)]

def has_emojis(text):
    if find_emojis(text):
        return True
    else:
        return False

In [ ]:
disaster_tweets_df_cleaned_RT['has_emojis'] = disaster_tweets_df_cleaned_RT['text'].apply(has_emojis)
disaster_tweets_df_cleaned_RT['emoji_count'] = disaster_tweets_df_cleaned_RT['text'].apply(find_emojis)
disaster_tweets_df_cleaned_RT['emoji_count'] = disaster_tweets_df_cleaned_RT['emoji_count'].apply(len)
disaster_tweets_df_cleaned_RT.sample(5)

,id,keyword,location,text,target,character_length,tokens_length,has_url,url_count,has_mentions,mentions_count,has_hashtags,hashtags_count,has_emojis,emoji_count
1170,1687,bridge%20collapse,NaN,Two giant cranes holding a bridge collapse into nearby homes http://t.co/OQpsvrGbJc,1,83,11,False,0,False,0,False,0,False,0
2405,3463,derailed,NaN,@jozerphine LITERALLY JUST LOOK THAT UP! YEAH DERAILED AT SMITHSONIAN SO EVERYTHIGN IS SHUT DOWN FROM FEDERAL CENTER SW TO MCPHERSON,1,132,21,False,0,True,1,False,0,False,0
2331,3355,demolition,USA,EPA begins demolition of homes in toxic area #Buffalo - http://t.co/noRkXBRS6G,1,78,11,False,0,False,0,True,1,False,0
7510,10743,wreckage,WorldWide,#Australia #News ; RT janeenorman: 'High probability' aircraft wreckage is from #MH370 according to Deputy Prime Û_ http://t.co/cdOHgnJmsT,1,139,18,False,0,False,0,True,3,False,0
3145,4519,emergency,NaN,STL Ace Grille - Surface Mounts SpeedTech Lights - Amber Emergency Lights - 544 http://t.co/t6Seku4yvm http://t.co/TJOZ4u4txl,0,125,16,False,0,False,0,False,0,False,0
